In [ ]:
import os, sys, importlib
sys.path.append(os.path.abspath(".."))
import json
import pandas as pd

## df_pre

In [ ]:

from utils import preprocess_listings
importlib.reload(preprocess_listings)
from utils.preprocess_listings import preprocess_host_variables, preprocess_obj_vars


RAW_DATA_DIR="../data_raw"
time_map={   
    '2306':"2309",
    '2309':"2312",
    '2312':"2403",
}
cities=['paris',"london"]

groups={f"{city}_{time}" :f"{city}_{time_nextQ}"for city in cities for time, time_nextQ in time_map.items()}
print(groups)


OUTPUT_FOLDER="../data_processed"
for i, (group, group_nextQ) in enumerate(groups.items()):
    city=group.split('_')[0]
    time=group.split('_')[1]
    print(i, city, time, group_nextQ)
    path_df=os.path.join(RAW_DATA_DIR, f"listings_{group}.csv") #"../data_raw\listings_paris2309.csv"
    path_df_nextQ=os.path.join(RAW_DATA_DIR, f"listings_{group_nextQ}.csv")#"../data_raw\listings_paris2312.csv"
    print(path_df,"exists:", os.path.exists(path_df))
    print(path_df_nextQ,"exists:", os.path.exists(path_df_nextQ))
    try:
        
        df=pd.read_csv(path_df)
        df_nextQ=pd.read_csv(path_df_nextQ)
        print(f"{i}/{group}: {df.shape}, {df_nextQ.shape}\n")

        df['in_paris']=1 if city=="paris" else 0
        df['time']=time

        df_filtered=preprocess_obj_vars(df=df, 
                    df_nextQ=df_nextQ, time=f"{time}",#***
                    proxy_vars=['price',"availability_30","availability_90"], 
                    get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
                    get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
                    obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
                    threshold_km=1, 
                    save=False,
                    output_folder=os.path.join(OUTPUT_FOLDER,f"{group}"), ##***
                    filename=f"listings_filtered_{group}.csv"#***
        )

        df_processed=preprocess_host_variables(df_raw=df_filtered, 
                                    save=True, output_folder=os.path.join(OUTPUT_FOLDER,f"{group}"), 
                                    filename=f"listings_processed_{group}.csv")

    except Exception as e:
        print(f"[error] in {group}!!")
        continue
    


{'paris_2306': 'paris_2309', 'paris_2309': 'paris_2312', 'paris_2312': 'paris_2403', 'london_2306': 'london_2309', 'london_2309': 'london_2312', 'london_2312': 'london_2403'}
0 paris 2306 paris_2309
../data_raw\listings_paris_2306.csv exists: True
../data_raw\listings_paris_2309.csv exists: True
0/paris_2306: (61706, 75), (67942, 75)



==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_

C:\Users\yeliu\AppData\Local\Temp\ipykernel_3936\3035214394.py:31: DtypeWarning: Columns (6,22,41,45,46,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df_nextQ=pd.read_csv(path_df_nextQ)


1/paris_2309: (67942, 75), (74330, 76)



==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique 

C:\Users\yeliu\AppData\Local\Temp\ipykernel_3936\3035214394.py:30: DtypeWarning: Columns (6,22,41,45,46,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path_df)


2/paris_2312: (74330, 76), (84397, 75)



==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique 

...

is_within_1km
0    68370
1     5960
Name: count, dtype: int64
# -----------------------filter & desc-------------------------
[INFO] vars to dropna:room_type; number_of_reviews_nextQ; availability_30; is_within_1km; instant_bookable; booking_rate_l90d; property_type; minimum_nights; availability_90; price
[CHECK] no filter on 'number_of_reviews_l30d','availability_30','booking_rate_l30d'!
[INFO]2 nan dropped in room_type
[INFO]6666 nan dropped in number_of_reviews_nextQ
[INFO]0 nan dropped in is_within_1km
[INFO]0 nan dropped in instant_bookable
[INFO]31279 nan dropped in booking_rate_l90d
[INFO]0 nan dropped in property_type
[INFO]0 nan dropped in minimum_nights
[INFO]0 nan dropped in availability_90
[INFO]554 nan dropped in price

filter by: room_type; number_of_reviews_nextQ; availability_30; is_within_1km; instant_bookable; booking_rate_l90d; property_type; minimum_nights; availability_90; price
len BEFORE: 74330
len AFTER: 35829


================= BALN PROCESSED VARIABLES =======

C:\Users\yeliu\AppData\Local\Temp\ipykernel_3936\3035214394.py:31: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df_nextQ=pd.read_csv(path_df_nextQ)


3/london_2306: (81791, 75), (87946, 75)



==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique

C:\Users\yeliu\AppData\Local\Temp\ipykernel_3936\3035214394.py:30: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path_df)


4/london_2309: (87946, 75), (91778, 75)



==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique

In [23]:

RAW_DATA_DIR="../data_raw"
time_map={   
    '2403':"2406",
}
cities=['paris',"london"]

groups={f"{city}_{time}" :f"{city}_{time_nextQ}"for city in cities for time, time_nextQ in time_map.items()}
print(groups)


OUTPUT_FOLDER="../data_processed"
for i, (group, group_nextQ) in enumerate(groups.items()):
    city=group.split('_')[0]
    time=group.split('_')[1]
    print(i, city, time, group_nextQ)
    path_df=os.path.join(RAW_DATA_DIR, f"listings_{group}.csv") #"../data_raw\listings_paris2309.csv"
    path_df_nextQ=os.path.join(RAW_DATA_DIR, f"listings_{group_nextQ}.csv")#"../data_raw\listings_paris2312.csv"
    print(path_df,"exists:", os.path.exists(path_df))
    print(path_df_nextQ,"exists:", os.path.exists(path_df_nextQ))
    try:
        
        df=pd.read_csv(path_df)
        df_nextQ=pd.read_csv(path_df_nextQ)
        print(f"{i}/{group}: {df.shape}, {df_nextQ.shape}\n")

        df['in_paris']=1 if city=="paris" else 0
        df['time']=time

        df_filtered=preprocess_obj_vars(df=df, 
                    df_nextQ=df_nextQ, time=f"{time}",#***
                    proxy_vars=['price',"availability_30","availability_90"], 
                    get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
                    get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
                    obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
                    threshold_km=1, 
                    save=False,
                    output_folder=os.path.join(OUTPUT_FOLDER,f"{group}"), ##***
                    filename=f"listings_filtered_{group}.csv"#***
        )

        df_processed=preprocess_host_variables(df_raw=df_filtered, 
                                    save=True, output_folder=os.path.join(OUTPUT_FOLDER,f"{group}"), 
                                    filename=f"listings_processed_{group}.csv")

    except Exception as e:
        print(f"[error] in {group}!!")
        continue
    


{'paris_2403': 'paris_2406', 'london_2403': 'london_2406'}
0 paris 2403 paris_2406
../data_raw\listings_paris_2403.csv exists: True
../data_raw\listings_paris_2406.csv exists: True
0/paris_2403: (84397, 75), (95885, 75)



==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f'

## 2023-09

十二月不够严谨，可能已经有pretrend？

In [10]:
path_df="../data_raw\listings_paris2309.csv"

df=pd.read_csv(path_df)
print(df.shape)

path_df_nextQ="../data_raw\listings_paris2312.csv"
df_nextQ=pd.read_csv(path_df_nextQ)
print(df_nextQ.shape)



(67942, 75)


C:\Users\yeliu\AppData\Local\Temp\ipykernel_3936\711757402.py:7: DtypeWarning: Columns (6,22,41,45,46,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df_nextQ=pd.read_csv(path_df_nextQ)


(74330, 76)


In [ ]:
## PARIS 2309
from utils import preprocess_listings
importlib.reload(preprocess_listings)
from utils.preprocess_listings import preprocess_host_variables, preprocess_obj_vars

OUTPUT_FOLDER="../data_processed"
df_filtered=preprocess_obj_vars(df=df, 
            df_nextQ=df_nextQ, time='2309',
            proxy_vars=['price',"availability_30","availability_90"], 
            get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
            get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
            obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
            threshold_km=1, 
            save=False,
            output_folder=os.path.join(OUTPUT_FOLDER,"paris_2309"), ##***
            filename=f"listings_filtered_paris_2309.csv"#***
)

df_processed=preprocess_host_variables(df_raw=df_filtered, 
                            save=True, output_folder=os.path.join(OUTPUT_FOLDER,"paris_2309"), 
                            filename=f"listings_processed_paris_2309.csv")




==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique :room_type, property_type, minimum_night

In [12]:
## LONDON 2309
path_df="../data_raw\listings_london2309.csv"

df=pd.read_csv(path_df)
print(df.shape)

path_df_nextQ="../data_raw\listings_london2312.csv"
df_nextQ=pd.read_csv(path_df_nextQ)
print(df_nextQ.shape)


C:\Users\yeliu\AppData\Local\Temp\ipykernel_3936\1053935669.py:4: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path_df)


(87946, 75)
(91778, 75)


In [14]:

from utils import preprocess_listings
importlib.reload(preprocess_listings)
from utils.preprocess_listings import preprocess_host_variables, preprocess_obj_vars

OUTPUT_FOLDER="../data_processed"
df_filtered=preprocess_obj_vars(df=df, 
            df_nextQ=df_nextQ, time=2309,
            proxy_vars=['price',"availability_30","availability_90"], 
            get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
            get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
            obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
            threshold_km=1, 
            save=False,
            output_folder=os.path.join(OUTPUT_FOLDER,"london_2309"), ##***
            filename=f"listings_filtered_london_2309.csv"#***
)

df_processed=preprocess_host_variables(df_raw=df_filtered, 
                            save=True, output_folder=os.path.join(OUTPUT_FOLDER,"london_2309"), 
                            filename=f"listings_processed_london_2309.csv")




==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique :room_type, property_type, minimum_night

## 2312

In [5]:
path_df="../data_raw\listings_paris2312.csv"
df=pd.read_csv(path_df)
print(df.shape)

path_df_nextQ="../data_raw\listings_paris2403.csv"
df_nextQ=pd.read_csv(path_df_nextQ)
print(df_nextQ.shape)


C:\Users\yeliu\AppData\Local\Temp\ipykernel_3936\2611826565.py:2: DtypeWarning: Columns (6,22,41,45,46,61) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(path_df)


(74330, 76)
(84397, 75)


In [6]:
## PARIS 2306
from utils import preprocess_listings
importlib.reload(preprocess_listings)
from utils.preprocess_listings import preprocess_host_variables, preprocess_obj_vars

OUTPUT_FOLDER="../data_processed"

df_filtered=preprocess_obj_vars(df=df, 
            df_nextQ=df_nextQ, time=2312,
            proxy_vars=['price',"availability_30","availability_90"], 
            get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
            get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
            obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
            threshold_km=1, 
            save=False,
            output_folder=os.path.join(OUTPUT_FOLDER,"paris_2312"), ##***
            filename=f"listings_filtered_paris_2312.csv"#***
)

df_processed=preprocess_host_variables(df_raw=df_filtered, 
                            save=True, output_folder=os.path.join(OUTPUT_FOLDER,"paris_2312"), 
                            filename=f"listings_processed_paris_2312.csv")

# df_processed=preprocess_host_variables(df)




==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique :room_type, property_type, minimum_night

d:\Edu\SelfPresentation_Multimodal_airbnb\utils\preprocess_listings.py:458: RuntimeWarning: All-NaN axis encountered
  # inter


is_within_1km
0    68370
1     5960
Name: count, dtype: int64
# -----------------------filter & desc-------------------------
[INFO] vars to dropna:room_type; number_of_reviews_nextQ; availability_30; is_within_1km; instant_bookable; booking_rate_l90d; property_type; minimum_nights; availability_90; price
[CHECK] no filter on 'number_of_reviews_l30d','availability_30','booking_rate_l30d'!
[INFO]2 nan dropped in room_type
[INFO]6666 nan dropped in number_of_reviews_nextQ
[INFO]0 nan dropped in is_within_1km
[INFO]0 nan dropped in instant_bookable
[INFO]31279 nan dropped in booking_rate_l90d
[INFO]0 nan dropped in property_type
[INFO]0 nan dropped in minimum_nights
[INFO]0 nan dropped in availability_90
[INFO]554 nan dropped in price

filter by: room_type; number_of_reviews_nextQ; availability_30; is_within_1km; instant_bookable; booking_rate_l90d; property_type; minimum_nights; availability_90; price
len BEFORE: 74330
len AFTER: 35829


================= BALN PROCESSED VARIABLES =======

In [7]:
path_df="../data_raw\listings_london2312.csv"
df=pd.read_csv(path_df)
print(df.shape)

path_df_nextQ="../data_raw\listings_london2403.csv"
df_nextQ=pd.read_csv(path_df_nextQ)
print(df_nextQ.shape)

(91778, 75)
(90852, 75)


In [8]:
## PARIS 2312
from utils import preprocess_listings
importlib.reload(preprocess_listings)
from utils.preprocess_listings import preprocess_host_variables, preprocess_obj_vars

OUTPUT_FOLDER="../data_processed"

df_filtered=preprocess_obj_vars(df=df, 
            df_nextQ=df_nextQ, time=2312,
            proxy_vars=['price',"availability_30","availability_90"], 
            get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
            get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
            obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
            threshold_km=1, 
            save=False,
            output_folder=os.path.join(OUTPUT_FOLDER,"london_2312"), ##***
            filename=f"listings_filtered_london_2312.csv"#***
)

df_processed=preprocess_host_variables(df_raw=df_filtered, 
                            save=True, output_folder=os.path.join(OUTPUT_FOLDER,"london_2312"), 
                            filename=f"listings_processed_london_2312.csv")




==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique :room_type, property_type, minimum_night

## 2306

In [ ]:
## PARIS 2306
from utils import preprocess_listings
importlib.reload(preprocess_listings)
from utils.preprocess_listings import preprocess_host_variables, preprocess_obj_vars

OUTPUT_FOLDER="../data_processed"

df_filtered=preprocess_obj_vars(df=df, 
            df_nextQ=df_nextQ, time=2312,# ??
            proxy_vars=['price',"availability_30","availability_90"], 
            get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
            get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
            obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
            threshold_km=1, 
            save=False,
            output_folder=os.path.join(OUTPUT_FOLDER,"paris_2312"), ##***
            filename=f"listings_filtered_paris_2312.csv"#***
)

df_processed=preprocess_host_variables(df_raw=df_filtered, 
                            save=True, output_folder=os.path.join(OUTPUT_FOLDER,"paris_2312"), 
                            filename=f"listings_processed_paris_2312.csv")

# df_processed=preprocess_host_variables(df)


In [ ]:
# LONDON 2306
OUTPUT_FOLDER="../data_processed"
df=pd.read_csv("../data_raw\listings_london2306.csv")
df_nextQ=pd.read_csv("../data_raw\listings_london2309.csv")

print(df.shape, df_nextQ.shape)

df_filtered=preprocess_obj_vars(df=df, 
            df_nextQ=df_nextQ, 
            proxy_vars=['price',"availability_30","availability_90"], 
            get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
            get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
            obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
            threshold_km=1, 
            save=False,
            output_folder=os.path.join(OUTPUT_FOLDER,"london_2306"), 
            filename=f"listings_filtered_london_2306.csv"#***
)

df_processed=preprocess_host_variables(df_raw=df_filtered, 
                            save=True, output_folder=os.path.join(OUTPUT_FOLDER,"london_2306"), 
                            filename=f"listings_processed_london_2306_{len(df_filtered)}.csv")


C:\Users\yeliu\AppData\Local\Temp\ipykernel_33372\3479131621.py:3: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df_nextQ=pd.read_csv("../data_raw\listings_london2309.csv")


(81791, 75) (87946, 75)


==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique :room_type, prop

## check 2406

In [ ]:
## paris 2406
OUTPUT_FOLDER="../data_processed"
df=pd.read_csv("../data_raw\listings_paris2406.csv")
df_nextQ=pd.read_csv("../data_raw\listings_paris2409.csv")

print(df.shape, df_nextQ.shape)

df_filtered=preprocess_obj_vars(df=df, 
            df_nextQ=df_nextQ, 
            proxy_vars=['price',"availability_30","availability_90"], 
            get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
            get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
            obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
            threshold_km=1, 
            save=False
)

df_processed=preprocess_host_variables(df_raw=df_filtered, 
                            save=True, output_folder=os.path.join(OUTPUT_FOLDER,"paris_2406"), 
                            filename=f"listings_processed_paris_2406_{len(df_filtered)}.csv")


(95885, 75) (95461, 75)


==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique :room_type, prop

In [ ]:
## LONDON 2406
OUTPUT_FOLDER="../data_processed"
df=pd.read_csv("../data_raw\listings_london2406.csv")
df_nextQ=pd.read_csv("../data_raw\listings_london2409.csv")

print(df.shape, df_nextQ.shape)

df_filtered=preprocess_obj_vars(df=df, 
            df_nextQ=df_nextQ, 
            proxy_vars=['price',"availability_30","availability_90"], 
            get_booking_rate_l30d=False, filtrate_by_booking_rate_l30d=False,#无输入时默认不按照booking筛选 
            get_booking_rate_l90d=True, filtrate_by_booking_rate_l90d=True,
            obj_vars=["room_type", 'property_type',"minimum_nights","instant_bookable"], 
            threshold_km=1, 
            save=False
)

df_processed=preprocess_host_variables(df_raw=df_filtered, 
                            save=True, output_folder=os.path.join(OUTPUT_FOLDER,"london_2406"), 
                            filename=f"listings_processed_london_2406_{len(df_filtered)}.csv")


(93481, 75) (96182, 75)


==========================PROXY + OBJ VARS============================
PROCESS PIPELINE :
1) process proxies : 
- price : delete '$', to_numeric

 2) if get_boooking_rate_l30d==True, calculation method:
- booking_rate_l30d = number_of_reviews_l30d / availability_30 
 note that many 'number_of_reviews_l30d' is 0!
 if availability_30 = 0, take NaN.
 if booking_rate_l30d > 1, take 1.

3) if 'add_booking_rate_l90d':
 ADD number_of_reviews_nextQ, booking_rate_l90d 
 if host no longer exists in df_nextQ / substraction get negative value/ ava_90_thisQ==0, then number_of_reviews_nextQ=> nan 
 1 >= booking_rate_l30d  = number_of_reviews_nextQ (Q3)/ availability_90 (Q2)
 if filter False: no filter on booking_rate, ava, nb_reviews

4) obj vars :
 - instant_bookable : fillna('f')
- minimum_nights : to_numeric, fillna(0)
- property_type : clean col : entire, hotel, shared, private, others.

5) filter : dropna on vars ==> desc df_filtered 

desc statistique :room_type, prop